In [ ]:
!pip install monai itk einops nibabel torch

In [ ]:
import torch
import monai
import monai.transforms as mt
from torch.utils.data import DataLoader, random_split
from monai.losses import DiceLoss
from monai.metrics import DiceMetric
from monai.transforms import AsDiscrete
from torch.utils.data import DataLoader, random_split
from pathlib import Path
from tqdm import tqdm
from google.colab import drive
from monai.networks.nets import UNETR

In [ ]:
drive.mount('/content/drive')

IMAGES_DIR = "/content/drive/MyDrive/panther/ImagesTr"
LABELS_DIR = "/content/drive/MyDrive/panther/LabelsTr"
OUTPUT_DIR = "/content/drive/MyDrive/MAE_models/"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
class SegmentationDataSet(monai.data.Dataset):
    def __init__(self, imagesTr, labelsTr):
        images = sorted(Path(imagesTr).glob("*.mha"))
        labels = sorted(Path(labelsTr).glob("*.mha"))
        data = [{"image": str(img), "label": str(lbl)} for img, lbl in zip(images, labels)]

        transforms = mt.Compose([
            mt.LoadImaged(keys=["image", "label"], reader="ITKReader"),
            mt.EnsureChannelFirstd(keys=["image", "label"]),
            mt.Spacingd(keys=["image", "label"], pixdim=(1.0, 1.0, 1.0), mode=["bilinear", "nearest"]),
            mt.Orientationd(keys=["image", "label"], axcodes="RAS"),
            mt.NormalizeIntensityd(keys=["image"]),
            mt.RandCropByPosNegLabeld(
                keys=["image", "label"], label_key="label",
                spatial_size=(96, 96, 96), pos=3, neg=1, num_samples=4,
            ),
            mt.RandFlipd(keys=["image", "label"], prob=0.2, spatial_axis=0),
            mt.RandFlipd(keys=["image", "label"], prob=0.2, spatial_axis=1),
            mt.RandFlipd(keys=["image", "label"], prob=0.2, spatial_axis=2),
            mt.RandRotate90d(keys=["image", "label"], prob=0.2, max_k=3),
            mt.RandScaleIntensityd(keys=["image"], factors=0.1, prob=0.2),
            mt.RandShiftIntensityd(keys=["image"], offsets=0.1, prob=0.2),
        ])
        super().__init__(data=data, transform=transforms)

In [ ]:
def train(epochs, lr):

      device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
      print(f"Using device: {device}")
      dataset = SegmentationDataSet(IMAGES_DIR, LABELS_DIR)
      torch.manual_seed(42)
      train_size = int(0.85 * len(dataset))
      val_size = len(dataset) - train_size
      train_dataset, val_dataset = random_split(dataset, [train_size, val_size])
      train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True, num_workers=2)
      val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, num_workers=2)
      mae_weights = torch.load("/content/drive/MyDrive/MAE_models/mae_model_phase1_v1.pth", map_location="cpu")

      model = UNETR(
          in_channels=1,
          out_channels=3,
          img_size=(96, 96, 96),
          feature_size=16,
          hidden_size=768,
          mlp_dim=3072,
          num_heads=12,
          spatial_dims=3,
      )

      unetr_weights = model.state_dict()
      matched = 0
      for key, value in mae_weights.items():
          new_key = f"vit.{key}"
          if new_key in unetr_weights and unetr_weights[new_key].shape == value.shape:
              unetr_weights[new_key] = value
              matched += 1

      model.load_state_dict(unetr_weights)
      print(f"Loaded {matched} layers from MAE")
      model = model.to(device)
      optimizer = torch.optim.Adam(model.parameters(), lr=lr)
      scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
          optimizer, T_max=epochs
      )

      loss_fn = DiceLoss(
          to_onehot_y=True,
          softmax=True,
          weight=torch.tensor([0.02, 0.85, 0.13]).to(device)
      )
      post_pred = AsDiscrete(argmax=True, to_onehot=3)
      post_label = AsDiscrete(to_onehot=3)
      dice_metric = DiceMetric(include_background=False, reduction="mean_batch")

      best_dice = 0.0

      for epoch in range(epochs):

          model.train()
          epoch_loss = 0.0

          for batch in tqdm(train_loader, desc=f"Epoch {epoch + 1}/{epochs}"):
              for sample in batch:
                  image = sample["image"].to(device)  # [1, 1, 96, 96, 96]
                  label = sample["label"].to(device)

                  optimizer.zero_grad()
                  output = model(image)
                  loss = loss_fn(output, label)
                  loss.backward()
                  optimizer.step()
                  epoch_loss += loss.item()

          avg_loss = epoch_loss / len(train_loader)

          model.eval()
          with torch.no_grad():
              for batch in val_loader:
                  for sample in batch:
                      image = sample["image"].to(device)
                      label = sample["label"].to(device)
                      output = model(image)

                      output_post = post_pred(output[0])
                      label_post = post_label(label[0])
                      dice_metric(y_pred=output_post.unsqueeze(0), y=label_post.unsqueeze(0))

          dice_per_class = dice_metric.aggregate()
          dice_pancreas = dice_per_class[1].item()
          dice_tumor = dice_per_class[0].item()
          dice_metric.reset()

          print(
              f"Epoch {epoch + 1}/{epochs}:\n Loss: {avg_loss:.4f} \n Dice pancreatic: {dice_pancreas:.4f} \n Dice tumor: {dice_tumor:.4f}")

          if dice_tumor > best_dice:
              best_dice = dice_tumor
              torch.save(model.state_dict(), "/content/drive/MyDrive/MAE_models/UNETR_MAE_phase2_v1.pth")
              print(f"Model saved (Dice guz: {best_dice:.4f})")

          scheduler.step()


In [ ]:
train(epochs=300, lr=1e-4)

Using device: cuda
Loaded 183 layers from MAE


Epoch 1/300: 100%|██████████| 39/39 [01:21<00:00,  2.08s/it]


Epoch 1/300:
 Loss: 1.2532 
 Dice pancreatic: 0.2080 
 Dice tumor: 0.0911
Model saved (Dice guz: 0.0911)


Epoch 2/300: 100%|██████████| 39/39 [01:23<00:00,  2.15s/it]


Epoch 2/300:
 Loss: 1.2335 
 Dice pancreatic: 0.2336 
 Dice tumor: 0.0884


...

Epoch 298/300:
 Loss: 0.8569 
 Dice pancreatic: 0.4593 
 Dice tumor: 0.3572


Epoch 299/300: 100%|██████████| 39/39 [01:21<00:00,  2.10s/it]


Epoch 299/300:
 Loss: 0.8164 
 Dice pancreatic: 0.4593 
 Dice tumor: 0.3572


Epoch 300/300: 100%|██████████| 39/39 [01:21<00:00,  2.08s/it]


Epoch 300/300:
 Loss: 0.8105 
 Dice pancreatic: 0.4593 
 Dice tumor: 0.3572
